### Build a Simple workflow
- State is the central data structure in LangGraph that is passed between nodes. 
- Each node reads from the state and returns updates to it.
- It serves as the shared memory for a graph execution and enables communication between nodes. 
- State is typically defined using TypedDict, Pydantic models, or MessagesState.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    """Represents the state of the graph."""
    graph_info: str
   

Nodes- nodes are the individual steps (functions) that perform work
Each node:

- Receives the current state
- Does some work
- Returns updates to the state

In [ ]:
def start_play(state: State):
    print("start_play node has been called.")
    return{"graph_info": state["graph_info"]+ "I'm starting to play. "}


def play_Soccer(state: State):
    print("play_Soccer node has been called.")
    return{"graph_info": state["graph_info"]+ "Soccer is fun! "}

def play_Basketball(state: State):
    print("play_Basketball node has been called.")
    return{"graph_info": state["graph_info"]+ "Basketball is fun! "}

In [ ]:
import random
from typing import Literal
def random_choice(state: State)->Literal["play_Soccer", "play_Basketball"]:
    graph_info =state["graph_info"]
    
    if random.random() < 0.5:
        return "play_Soccer"
    else:
        return "play_Basketball"

### Graph Construction

1. Define State
   → Define the data that flows through the graph.

2. Create Nodes
   → Write functions that perform work and update state.

3. Create Graph Builder
   → Initialize a StateGraph with the state schema.

4. Add Nodes
   → Register node functions with the graph.

5. Add Edges / Conditional Edges
   → Define the execution flow between nodes.

6. Compile
   → Convert the graph definition into an executable graph.

7. Invoke
   → Run the graph with an initial state input.


- we can visualize the graph using Mermaid diagram

In [ ]:
from IPython.display import Image, display

from langgraph.graph import StateGraph, START, END

# Build the graph
graph = StateGraph(State)

# Add nodes
graph.add_node("start_play", start_play)
graph.add_node("play_Soccer", play_Soccer)
graph.add_node("play_Basketball", play_Basketball)

# Add edges
graph.add_edge(START, "start_play")
graph.add_conditional_edges("start_play", random_choice)
graph.add_edge("play_Soccer", END)
graph.add_edge("play_Basketball", END)

# Compile the graph
graph_builder = graph.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))

### Graph Invocation

In [ ]:
graph_builder.invoke({"graph_info": "I'm John. "})